# 001 Source Inventory And Safety Audit

Forensic checkpoint before preprocessing. This notebook inventories acquired public data, audits license/safety posture, visualizes source coverage, and writes source-quality artifacts. It does **not** normalize, train, index, or build the assistant.

## Audit Outputs

Expected outputs under `data/processed/source_inventory_audit/`:

- `source_file_inventory.csv`
- `source_run_inventory.csv`
- `artifact_status_inventory.csv`
- `source_quality_matrix.csv`
- `source_quality_matrix.md`
- `source_quality_matrix.json`
- `lightweight_risk_screening.csv`

In [1]:
from pathlib import Path
import json, re
from collections import Counter

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
except ModuleNotFoundError:
    plt = sns = None
    HAS_PLOTS = False

try:
    from IPython.display import display, Markdown
except ModuleNotFoundError:
    Markdown = str
    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / 'data'
RAW = DATA / 'raw'
PROCESSED = DATA / 'processed'
OUT = PROCESSED / 'source_inventory_audit'
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 180)
if HAS_PLOTS:
    sns.set_theme(style='whitegrid')
else:
    print('Plotting libraries are not installed; audit tables/files will still be produced. Install requirements.txt for charts.')

PROJECT_ROOT

Plotting libraries are not installed; audit tables/files will still be produced. Install requirements.txt for charts.


WindowsPath('c:/Users/nwagb/Desktop/it_support_ai')

In [ ]:
registry = json.loads((DATA / 'source_registry.json').read_text(encoding='utf-8'))
families = pd.DataFrame(registry['source_families'])
runs = pd.DataFrame(registry['acquisition_runs'])

print(registry['project_boundary'])
print(registry['public_data_boundary'])
print(registry['cyber_safety_boundary'])
display(families[['id', 'source_type', 'commercial_posture', 'cyber_safety_posture', 'license']].sort_values('id'))

## Raw File Inventory

This inventories what landed on disk. It intentionally keeps archives untouched.

In [ ]:
def compound_suffix(path: Path) -> str:
    name = path.name.lower()
    for suffix in ('.tar.gz', '.json.gz', '.xml.gz'):
        if name.endswith(suffix):
            return suffix
    return path.suffix.lower() or '<none>'

rows = []
for path in RAW.rglob('*'):
    if not path.is_file():
        continue
    rel = path.relative_to(PROJECT_ROOT)
    parts = rel.parts
    rows.append({
        'path': str(rel),
        'source_family_id': parts[2] if len(parts) > 2 else '<unknown>',
        'run_id': parts[3] if len(parts) > 3 else '<none>',
        'filename': path.name,
        'extension': compound_suffix(path),
        'bytes': path.stat().st_size,
        'mb': path.stat().st_size / 1_048_576,
        'modified': pd.Timestamp(path.stat().st_mtime, unit='s'),
    })

files = pd.DataFrame(rows).sort_values(['source_family_id', 'run_id', 'path'])
SOURCE_IDS = set(families['id'])
source_files = files[files['source_family_id'].isin(SOURCE_IDS)].copy()
files.to_csv(OUT / 'source_file_inventory.csv', index=False)
print(f'files: {len(files):,}')
print(f'source files: {len(source_files):,}')
display(files.head(20))

In [ ]:
source_file_summary = (
    source_files.groupby('source_family_id', as_index=False)
    .agg(files=('path', 'count'), total_mb=('mb', 'sum'), extensions=('extension', lambda x: ', '.join(sorted(set(x)))))
    .merge(families[['id', 'source_type', 'commercial_posture', 'cyber_safety_posture']], left_on='source_family_id', right_on='id', how='left')
    .drop(columns=['id'])
    .sort_values('total_mb', ascending=False)
)
display(source_file_summary)

In [ ]:
ext_summary = source_files.groupby('extension', as_index=False).agg(files=('path', 'count'), total_mb=('mb', 'sum')).sort_values('files', ascending=False)
if HAS_PLOTS and len(source_file_summary):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.barplot(data=source_file_summary, y='source_family_id', x='total_mb', ax=axes[0], color='#4C78A8')
    axes[0].set_title('Raw Data Size By Source')
    axes[0].set_xlabel('MB')
    axes[0].set_ylabel('')

    sns.barplot(data=ext_summary.head(15), x='files', y='extension', ax=axes[1], color='#59A14F')
    axes[1].set_title('Top File Extensions')
    axes[1].set_xlabel('Files')
    axes[1].set_ylabel('')
    plt.tight_layout()
    plt.show()
display(ext_summary)

## Acquisition Run And Artifact Status

Run-level status comes from `source_registry.json`; artifact-level status comes from broad-wave manifests.

In [ ]:
run_cols = ['run_id', 'source_family_id', 'status', 'raw_dir', 'artifact_count', 'successful_artifact_count', 'question_count', 'attribution_record_count', 'report_path']
run_inventory = runs.reindex(columns=run_cols).copy()
run_inventory.to_csv(OUT / 'source_run_inventory.csv', index=False)
display(run_inventory.sort_values(['run_id', 'source_family_id']))

In [ ]:
artifact_rows = []
for manifest_path in (RAW / 'broad_data_wave').glob('*/run_manifest.json'):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    for artifact in manifest.get('artifacts', []):
        artifact_rows.append({'run_id': manifest['run_id'], **artifact})

artifacts = pd.DataFrame(artifact_rows)
artifacts.to_csv(OUT / 'artifact_status_inventory.csv', index=False)
artifact_status = artifacts.groupby(['run_id', 'source_id', 'status'], as_index=False).size() if len(artifacts) else pd.DataFrame()
display(artifact_status)

In [ ]:
if len(artifacts) and HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(14, 5))
    plot_df = artifact_status.rename(columns={'size': 'artifacts'})
    sns.barplot(data=plot_df, y='source_id', x='artifacts', hue='status', ax=ax)
    ax.set_title('Broad-Wave Artifact Status By Source')
    ax.set_xlabel('Artifacts')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

if len(artifacts):
    failed = artifacts[artifacts['status'].eq('failed')][['run_id', 'source_id', 'kind', 'url', 'error']]
    
    display(Markdown('### Failed Or Gated Artifacts'))
    display(failed if len(failed) else pd.DataFrame({'status': ['no failures']}))

## Source Quality Matrix

This is the gate artifact. Downstream notebooks should use this matrix before extraction, normalization, indexing, or training.

In [ ]:
acquired = set(runs.loc[runs['status'].isin(['raw_acquired', 'raw_inspected']), 'source_family_id'])

def risk_and_use(row):
    sid, stype, posture = row['id'], row['source_type'], row['cyber_safety_posture']
    if sid == 'public_it_helpdesk_ticket_datasets':
        return 'high_until_reviewed', 'inspect_only_pending_pii_license_schema', 'blocked_until_pii_license_review'
    if posture == 'defensive_metadata_only':
        return 'low_pii_high_safety_boundary', 'defensive_metadata_retrieval_and_evaluation_only', 'allowed_after_metadata_schema_review'
    if posture == 'security_records_require_review':
        return 'medium_pii_medium_safety', 'classifier_and_retrieval_after_security_filtering', 'requires_record_level_safety_filter'
    if posture == 'defensive_vendor_metadata_only':
        return 'low_pii_medium_terms', 'defensive_firmware_patch_retrieval_and_evaluation', 'allowed_after_terms_and_metadata_review'
    if stype in {'documentation', 'manuals', 'developer_documentation', 'vendor_documentation_source_repo'}:
        return 'low', 'retrieval_reference_and_evaluation', 'allowed_after_license_path_review'
    if stype == 'repair_guides':
        return 'low', 'poc_retrieval_reference_only', 'allowed_for_poc_after_license_review'
    return 'medium', 'inspect_before_use', 'requires_review'

matrix_rows = []
for _, row in families.sort_values('id').iterrows():
    pii_risk, allowed_use, gate = risk_and_use(row)
    matrix_rows.append({
        'source_family_id': row['id'],
        'source_type': row['source_type'],
        'acquired_status': 'acquired' if row['id'] in acquired else 'candidate_not_acquired',
        'public_access_status': 'public_freely_accessible_only',
        'license_posture': row['license'],
        'commercial_posture': row['commercial_posture'],
        'attribution_required': row['attribution_required'],
        'share_alike_required': row['share_alike_required'],
        'cyber_safety_posture': row['cyber_safety_posture'],
        'pii_secrets_risk': pii_risk,
        'allowed_downstream_use': allowed_use,
        'gate_decision': gate,
    })

quality = pd.DataFrame(matrix_rows)
quality.to_csv(OUT / 'source_quality_matrix.csv', index=False)
quality.to_json(OUT / 'source_quality_matrix.json', orient='records', indent=2)
display(quality)

In [ ]:
def df_to_markdown(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for _, row in df.iterrows():
        vals = [str(row[c]).replace('|', '\\|').replace('\n', ' ') for c in cols]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines) + '\n'

(OUT / 'source_quality_matrix.md').write_text(df_to_markdown(quality), encoding='utf-8')

if HAS_PLOTS and len(quality):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.countplot(data=quality, y='cyber_safety_posture', ax=axes[0], color='#4C78A8')
    axes[0].set_title('Cyber Safety Posture')
    axes[0].set_xlabel('Sources')
    axes[0].set_ylabel('')
    sns.countplot(data=quality, y='pii_secrets_risk', ax=axes[1], color='#F28E2B')
    axes[1].set_title('PII / Secrets Risk')
    axes[1].set_xlabel('Sources')
    axes[1].set_ylabel('')
    sns.countplot(data=quality, y='gate_decision', ax=axes[2], color='#59A14F')
    axes[2].set_title('Downstream Gate Decision')
    axes[2].set_xlabel('Sources')
    axes[2].set_ylabel('')
    plt.tight_layout()
    plt.show()

## Lightweight Risk Screening

This is a small early-warning scan, not a final filter. It reads bounded text snippets from non-archive files only and reports counts, not sensitive content.

In [ ]:
TEXT_EXTS = {'.txt', '.md', '.json', '.jsonl', '.csv', '.xml', '.raw', '.html', '<none>'}
MAX_FILES_PER_SOURCE = 60
MAX_BYTES_PER_FILE = 2_000_000
PATTERNS = {
    'email_like': re.compile(r'\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b', re.I),
    'ipv4_like': re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b'),
    'secret_keyword': re.compile(r'\b(password|passwd|api[_-]?key|token|secret|private[_-]?key)\b', re.I),
    'unsafe_cyber_hint': re.compile(r'\b(exploit|payload|credential dump|phishing kit|malware sample|reverse shell|persistence|evasion)\b', re.I),
}

scan_rows = []
scan_candidates = (
    source_files[source_files['extension'].isin(TEXT_EXTS) & source_files['bytes'].between(1, MAX_BYTES_PER_FILE)]
    .groupby('source_family_id', group_keys=False)
    .head(MAX_FILES_PER_SOURCE)
)

for _, item in scan_candidates.iterrows():
    path = PROJECT_ROOT / item['path']
    try:
        text = path.read_text(encoding='utf-8', errors='ignore')[:MAX_BYTES_PER_FILE]
    except OSError as exc:
        scan_rows.append({'path': item['path'], 'source_family_id': item['source_family_id'], 'read_error': repr(exc)})
        continue
    row = {'path': item['path'], 'source_family_id': item['source_family_id'], 'chars_scanned': len(text), 'read_error': ''}
    row.update({name: len(pattern.findall(text)) for name, pattern in PATTERNS.items()})
    scan_rows.append(row)

risk_screen = pd.DataFrame(scan_rows)
risk_screen.to_csv(OUT / 'lightweight_risk_screening.csv', index=False)
risk_summary = risk_screen.groupby('source_family_id', as_index=False)[list(PATTERNS)].sum() if len(risk_screen) else pd.DataFrame()
display(risk_summary.sort_values(list(PATTERNS), ascending=False) if len(risk_summary) else risk_summary)

In [ ]:
if len(risk_screen) and HAS_PLOTS:
    plot = risk_summary.melt(id_vars='source_family_id', var_name='risk_signal', value_name='count')
    fig, ax = plt.subplots(figsize=(15, 6))
    sns.barplot(data=plot, y='source_family_id', x='count', hue='risk_signal', ax=ax)
    ax.set_title('Lightweight Text Risk Signals By Source')
    ax.set_xlabel('Pattern hits in bounded sample')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

if len(risk_screen):
    display(Markdown('### Files With Any Risk Signal'))
    display(risk_screen[risk_screen[list(PATTERNS)].sum(axis=1).gt(0)].sort_values(list(PATTERNS), ascending=False).head(50))

## Downstream Readiness Decision

Proceed to preprocessing only source-by-source. The next notebook should not consume raw data blindly; it should read `source_quality_matrix.csv` and enforce `gate_decision`.

In [ ]:
display(Markdown('### Gate Counts'))
display(quality.groupby(['gate_decision', 'allowed_downstream_use'], as_index=False).size().sort_values('size', ascending=False))

display(Markdown('### Recommended Next Actions'))
for line in [
    'Inspect public ticket datasets for PII, secrets, schema, labels, and license before any classifier use.',
    'Inspect MicrosoftDocs and MDN zip LICENSE files and documentation paths before extraction.',
    'Normalize Stack Exchange into question-centric records with security-tag filtering.',
    'Keep NVD, CISA, GitHub advisories as defensive metadata only.',
    'Do not begin model training, vector indexing, or app work until source-specific inspection artifacts exist.',
]:
    display(Markdown(f'- {line}'))

print(f'Wrote audit artifacts to: {OUT.relative_to(PROJECT_ROOT)}')